In [ ]:
!pip install plotly

In [ ]:
#табличка
import pandas as pd

url = 'https://docs.google.com/spreadsheets/d/1dMWPr-EKFXF0VwCppeFm5cMv5sWRVyCbyEz7fTHGHaw/edit?pli=1&gid=0#gid=0'
csv_url = url.split('/edit')[0] + '/export?format=csv&gid=0'
df = pd.read_csv(csv_url)
df.columns = ['Дата', 'Тип', 'Минимальная цена', 'Средняя цена', 'Ссылки']
print("✅ Загружено строк:", len(df))
df

✅ Загружено строк: 28


,Дата,Тип,Минимальная цена,Средняя цена,Ссылки
0,24.04.2026,Лофт,1700,"1966,666667",https://vk.com/wall-130344439_6003
1,13.02.2026,Лофт,1800,2000,https://vk.com/wall-130344439_5958
2,15.11.2025,Посвят,3500,"3733,333333",https://vk.com/wall-130344439_5928
3,05.09.2025,Лофт,1800,2000,https://vk.com/wall-130344439_5909
4,25.04.2025,Лофт,1500,1600,https://vk.com/wall-130344439_5836


In [ ]:
#преобразуем дату
df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')
df['Год'] = df['Дата'].dt.year

#чистим цены
for col in ['Минимальная цена', 'Средняя цена']:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = df[col].str.replace(r'[^\d\.\-]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

#удаляем строки без года или без средней цены
df_clean = df.dropna(subset=['Год', 'Средняя цена']).copy()
df_clean['Тип'] = df_clean['Тип'].astype(str).str.strip().str.lower()

#отдельные датафреймы для лофт и посвят
loft = df_clean[df_clean['Тип'] == 'лофт']
posvyat = df_clean[df_clean['Тип'] == 'посвят']

print(f"Всего записей: {len(df_clean)}")
print(f"Лофт: {len(loft)}, Посвят: {len(posvyat)}")
print("Годы:", sorted(df_clean['Год'].unique()))

Всего записей: 28
Лофт: 19, Посвят: 9
Годы: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


/tmp/ipykernel_13120/3519652471.py:2: UserWarning:

Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [ ]:
#средняя цена за все годы (все, лофт, посвят)

mean_all = df_clean['Средняя цена'].mean()
mean_loft = loft['Средняя цена'].mean() if not loft.empty else None
mean_posvyat = posvyat['Средняя цена'].mean() if not posvyat.empty else None

print("СРЕДНИЕ ЦЕНЫ ЗА ВСЕ ГОДЫ:")
print(f"  • Все мероприятия:        {mean_all:,.2f}")
if mean_loft is not None:
    print(f"  • Только «лофт»:           {mean_loft:,.2f}")
else:
    print("  • Только «лофт»:           нет данных")
if mean_posvyat is not None:
    print(f"  • Только «посвят»:         {mean_posvyat:,.2f}")


СРЕДНИЕ ЦЕНЫ ЗА ВСЕ ГОДЫ:
  • Все мероприятия:        2,054.62
  • Только «лофт»:           1,278.75
  • Только «посвят»:         3,692.59


In [ ]:
#общий график: средняя цена всех мероприятий (лофт+посвят) по годам
import plotly.express as px

total_yearly = df_clean.groupby('Год')['Средняя цена'].mean().reset_index()

fig = px.line(total_yearly, x='Год', y='Средняя цена',
              title='Общая средняя цена проходки (лофт + посвят) по годам',
              markers=True, line_shape='linear')
fig.update_traces(line_color='green', line_width=3, marker_size=10)
fig.update_layout(xaxis=dict(tickmode='linear'), hovermode='x unified')
fig.show()

In [ ]:
#сравнение динамики "лофт" и "посвят"
loft_yearly = loft.groupby('Год')['Средняя цена'].mean().reset_index()
loft_yearly['Тип'] = 'Лофт'
posvyat_yearly = posvyat.groupby('Год')['Средняя цена'].mean().reset_index()
posvyat_yearly['Тип'] = 'Посвят'
combined = pd.concat([loft_yearly, posvyat_yearly])

fig = px.line(combined, x='Год', y='Средняя цена', color='Тип',
              title='Сравнение динамики цен: Лофт vs Посвят',
              markers=True, line_shape='linear',
              color_discrete_map={'Лофт':'fuchsia', 'Посвят':'teal'})
fig.update_traces(line_width=3, marker_size=10)
fig.update_layout(xaxis=dict(tickmode='linear'), hovermode='x unified')
fig.show()

In [ ]:
#сравнение средней цены "лофт" и "посвят"
pivot = df_clean.groupby(['Год', 'Тип'])['Средняя цена'].mean().unstack()
pivot = pivot[['лофт', 'посвят']].rename(columns={'лофт':'Лофт', 'посвят':'Посвят'})

fig = px.bar(pivot, x=pivot.index, y=['Лофт', 'Посвят'],
             title='Сравнение средней цены: Лофт и Посвят по годам',
             barmode='group', text_auto='.2f', color_discrete_sequence=['fuchsia', 'teal'])
fig.update_layout(xaxis_title='Год', yaxis_title='Средняя цена',
                  legend_title='Тип мероприятия')
fig.show()

In [ ]:
#динамика средней цены "лофт" и "посвят"
yearly_stats = df_clean.groupby('Год').agg(
    Средняя_цена=('Средняя цена', 'mean'),
    Минимальная_цена=('Минимальная цена', 'mean')
).reset_index()

fig = px.line(yearly_stats, x='Год', y=['Средняя_цена', 'Минимальная_цена'],
              title='Динамика средней и минимальной цены (все мероприятия)',
              markers=True, labels={'value':'Цена', 'variable':'Показатель'},
              color_discrete_sequence=['pink', 'olive'])
fig.update_traces(line_width=2.5)
fig.show()

In [ ]:
# индексы лофтов и посвятов относительно прошлых (отношение друг к другу)

loft_1 = loft.sort_values('Дата').copy().reset_index(drop=True)
posvyat_1 = posvyat.sort_values('Дата').copy().reset_index(drop=True)

def dynamics(table):
    table['Динамика цен (мин.)'] = 100.0
    table['Динамика цен (средн.)'] = 100.0

    for i in range(1, len(table)):
        price_min_curr = table.loc[i, 'Минимальная цена']
        price_min_prev = table.loc[i-1, 'Минимальная цена']
        price_avg_curr = table.loc[i, 'Средняя цена']
        price_avg_prev = table.loc[i-1, 'Средняя цена']
        table.loc[i, 'Динамика цен (мин.)'] = price_min_curr/price_min_prev*100
        table.loc[i, 'Динамика цен (средн.)'] = price_avg_curr/price_avg_prev*100
    return table

loft_1 = dynamics(loft_1)
posvyat_1 = dynamics(posvyat_1)

# print(loft_1)
# print(posvyat_1)

In [ ]:
# экспортируем данные по ИПЦ

url = 'https://docs.google.com/spreadsheets/d/1qP7WtCuynOmA4hGCV9KTUFmobZnjJDeaqiBXliZ8CSQ/export?format=xlsx'

df_ipc = pd.read_excel(url)
df_ipc['ИПЦ'] = df_ipc['ИПЦ'].astype(str).str.replace(',', '.').astype(float)
df_ipc = df_ipc.dropna().reset_index(drop=True)

# print(df_ipc)

In [ ]:
# индексы лофтов и посвятов по годам + ИПЦ + реальная цена лофтов и посвятов

loft_index = loft_yearly.copy()
posvyat_index = posvyat_yearly.copy()
loft_index.loc[0,'индекс лофтов'] = 100.0
posvyat_index.loc[0,'индекс посвятов'] = 100.0

def indexes_yearly(table, index_name):
    for i in range(1,len(table)):
      table.loc[i,index_name] = 100*table.loc[i,'Средняя цена']/table.loc[i-1,'Средняя цена']
    return table

loft_index = indexes_yearly(loft_index, 'индекс лофтов')
posvyat_index = indexes_yearly(posvyat_index, 'индекс посвятов')

loft_index['ИПЦ'] = df_ipc['ИПЦ'].iloc[2:].values
posvyat_index['ИПЦ'] = df_ipc['ИПЦ']

loft_index.loc[0,'ИПЦ'] = 100.0
posvyat_index.loc[0,'ИПЦ'] = 100.0

loft_index['Накопленный_ИПЦ'] = (loft_index['ИПЦ']/100).cumprod()
posvyat_index['Накопленный_ИПЦ'] = (posvyat_index['ИПЦ']/100).cumprod()

loft_index['реальная цена'] = loft_index['Средняя цена']/(loft_index['Накопленный_ИПЦ'])
posvyat_index['реальная цена'] = posvyat_index['Средняя цена']/(posvyat_index['Накопленный_ИПЦ'])


print(loft_index)
print(posvyat_index)

    Год  Средняя цена   Тип  индекс лофтов     ИПЦ  Накопленный_ИПЦ  \
0  2019    887.500000  Лофт     100.000000  100.00         1.000000   
1  2020    875.000000  Лофт      98.591549  104.91         1.049100   
2  2021    991.666667  Лофт     113.333333  108.39         1.137119   
3  2022   1083.166667  Лофт     109.226891  111.94         1.272892   
4  2023   1166.666667  Лофт     107.708878  107.42         1.367340   
5  2024   1375.000000  Лофт     117.857143  109.52         1.497511   
6  2025   1735.000000  Лофт     126.181818  105.59         1.581222   
7  2026   1983.333333  Лофт     114.313160  102.97         1.628184   

   реальная цена  
0     887.500000  
1     834.048232  
2     872.086597  
3     850.949683  
4     853.238092  
5     918.190318  
6    1097.252806  
7    1218.126018  
    Год  Средняя цена     Тип  индекс посвятов     ИПЦ  Накопленный_ИПЦ  \
0  2017   3000.000000  Посвят       100.000000  100.00         1.000000   
1  2018   3200.000000  Посвят       106

In [ ]:
# лофты vs инфляция
import plotly.graph_objects as go
try:
    loft_index
except NameError:
    raise NameError("Переменная 'loft_index' не найдена. Выполните предыдущие ячейки.")
if 'Год' not in loft_index.columns:
    loft_index = loft_index.reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=loft_index['Год'], y=loft_index['индекс лофтов'],
                         mode='lines+markers', name='Лофты',
                         line=dict(color='fuchsia', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Лофты</b>: %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Scatter(x=loft_index['Год'], y=loft_index['ИПЦ'],
                         mode='lines+markers', name='ИПЦ (инфляция)',
                         line=dict(color='gray', width=2, dash='dash'), marker=dict(size=8, symbol='square'),
                         hovertemplate='<b>Год</b>: %{x}<br><b>ИПЦ</b>: %{y:.1f}%<extra></extra>'))
fig.update_layout(title='Динамика цен на лофты vs официальная инфляция',
                  xaxis_title='Год', yaxis_title='Индекс, % (год к году)',
                  hovermode='x unified', template='plotly_white',
                  xaxis=dict(tickmode='linear', tick0=loft_index['Год'].min(), dtick=1))
fig.show()

In [ ]:
# посвяты vs инфляция
try:
    posvyat_index
except NameError:
    raise NameError("Переменная 'posvyat_index' не найдена. Выполните предыдущие ячейки.")

if 'Год' not in posvyat_index.columns:
    posvyat_index = posvyat_index.reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=posvyat_index['Год'], y=posvyat_index['индекс посвятов'],
                         mode='lines+markers', name='Посвяты',
                         line=dict(color='teal', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Посвяты</b>: %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Scatter(x=posvyat_index['Год'], y=posvyat_index['ИПЦ'],
                         mode='lines+markers', name='ИПЦ (инфляция)',
                         line=dict(color='gray', width=2, dash='dash'), marker=dict(size=8, symbol='square'),
                         hovertemplate='<b>Год</b>: %{x}<br><b>ИПЦ</b>: %{y:.1f}%<extra></extra>'))
fig.update_layout(title='Динамика цен на посвят vs официальная инфляция',
                  xaxis_title='Год', yaxis_title='Индекс, % (год к году)',
                  hovermode='x unified', template='plotly_white',
                  xaxis=dict(tickmode='linear', tick0=posvyat_index['Год'].min(), dtick=1))
fig.show()

In [ ]:
# Реальные цены лофтов + посвятов
try:
    loft_index, posvyat_index
except NameError:
    raise NameError("Переменные 'loft_index' и 'posvyat_index' не найдены. Выполните предыдущие ячейки.")
for df in (loft_index, posvyat_index):
    if 'Год' not in df.columns:
        df.reset_index(inplace=True)
fig = go.Figure()
fig.add_trace(go.Scatter(x=loft_index['Год'], y=loft_index['реальная цена'],
                         mode='lines+markers', name='Лофты',
                         line=dict(color='fuchsia', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Лофты (реальная)</b>: %{y:,.0f} ₽<extra></extra>'))
fig.add_trace(go.Scatter(x=posvyat_index['Год'], y=posvyat_index['реальная цена'],
                         mode='lines+markers', name='Посвяты',
                         line=dict(color='teal', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Посвяты (реальная)</b>: %{y:,.0f} ₽<extra></extra>'))
fig.update_layout(title='Реальные цены лофтов и посвятов',
                  xaxis_title='Год', yaxis_title='Цена в ценах базового года, ₽',
                  hovermode='x unified', template='plotly_white',
                  xaxis=dict(tickmode='linear', dtick=1), legend=dict(x=0.02, y=0.98))
fig.show()

In [ ]:
# Ключевая ставка по данным ЦБ РФ (01.01.2017 - 31.12.2025 ) вклад=ставка*количество дней действия в году
data = {
    'Год': [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Средневзвешенная ключевая ставка, %': [9.89, 7.41, 7.31, 5.69, 5.72, 10.59, 9.61, 17.45, 19.13]
}

df_keyrate = pd.DataFrame(data)

print("Среднегодовая ключевая ставка ЦБ РФ (взвешенная по дням действия):")
print(df_keyrate.to_string(index=False))

Среднегодовая ключевая ставка ЦБ РФ (взвешенная по дням действия):
 Год  Средневзвешенная ключевая ставка, %
2017                                 9.89
2018                                 7.41
2019                                 7.31
2020                                 5.69
2021                                 5.72
2022                                10.59
2023                                 9.61
2024                                17.45
2025                                19.13


In [ ]:
import pandas as pd

sheet_id = "1l_ldocDQXMZFGB9S-usmkTlQeLsYcmVX"
url = f"https://docs.google.com/spreadsheets/d/1uV71Ef_ToavRgC4nPTBfi6jnyjm9e1s529X0ssZPe1Y/edit?usp=sharing"

# 1. Читаем CSV без parse_dates, чтобы узнать настоящие имена колонок
df = pd.read_csv(url, decimal=',')

# 2. Выводим реальные имена колонок
print("Реальные колонки в файле:", df.columns.tolist())

# 3. Приводим названия к нижнему регистру и удаляем лишние пробелы
df.columns = df.columns.str.strip().str.lower()

# 4. Определяем, какая колонка отвечает за дату, а какая за ставку
#    (можно подставить точные имена после просмотра вывода)
date_col = 'дата'      # если после приведения к нижнему регистру стало 'дата'
rate_col = 'ставка'    # если после приведения стало 'ставка'

# 5. Преобразуем колонку с датой
df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')

# 6. Удаляем строки с пустыми значениями
df = df.dropna(subset=[date_col, rate_col])

# 7. Сортируем по дате
df = df.sort_values(date_col)

# 8. Добавляем колонку "год"
df['год'] = df[date_col].dt.year

# 9. Группируем по годам и считаем статистику
yearly_stats = df.groupby('год').agg(
    ставка_начало_года = (rate_col, 'first'),
    ставка_конец_года  = (rate_col, 'last'),
    средняя_ставка     = (rate_col, 'mean'),
    мин_ставка         = (rate_col, 'min'),
    макс_ставка        = (rate_col, 'max')
).reset_index()

yearly_stats['средняя_ставка'] = yearly_stats['средняя_ставка'].round(2)

print("\n=== Ключевая ставка по годам ===")
print(yearly_stats.to_string(index=False))

Реальные колонки в файле: ['<!DOCTYPE html><html lang="en-US"><head><script nonce="PSqn-44QPz45raO1_11L4g">var DOCS_timing={}; DOCS_timing[\'pls\']=new Date().getTime();</script><meta property="og:title" content="ключевая ставка"><meta property="og:type" content="article"><meta property="og:site_name" content="Google Docs"><meta property="og:url" content="https://docs.google.com/spreadsheets/d/1uV71Ef_ToavRgC4nPTBfi6jnyjm9e1s529X0ssZPe1Y/edit?usp=sharing&amp;usp=embed_facebook"><meta property="og:image" content="https://lh7-us.googleusercontent.com/docs/AHkbwyL1_In5WQXCB63L_mK4XcqR7ivXsJ3w7ja4dKoHA6ONI6-zCx3DO2MkoBSOKgw85wG_gInI39OE12Z0FbCBacVvFGneoC1ihTotCMdTdkKCCZsFCBYI=w1200-h630-p"><meta property="og:image:width" content="1200"><meta property="og:image:height" content="630"><meta name="google" content="notranslate"><meta http-equiv="X-UA-Compatible" content="IE=edge;"><meta name="referrer" content="origin"><title>ключевая ставка - Google Sheets</title><link rel="shortcut icon" href

KeyError: 'дата'